# 🏗️ Week 11 hands-on: Evaluation harness and CI

## Learning Objectives
In this notebook, you will learn:
1. **Pins and versions** - hash every input a result depends on, and refuse to run when one changed
2. **The run artifact** - run a suite through earlier weeks' checks and write a checksummed `results.json`
3. **Workers and the cache** - run cases in parallel and from a cache without changing a result
4. **Budgets and exit codes** - stop a run at its budget and report it as incomplete, never as a pass
5. **Flaky results** - repeat a case with every input fixed and tell the judge's noise from the agent's

## Where this fits
This is the runnable companion to `week-11-evaluation-harness-and-ci-handson.md`, using the same code as `lab.py` (a test keeps the two identical). Read `week-11-evaluation-harness-and-ci-theory.md` first. Weeks 7 and 10 built the checks. This week runs them as a harness whose results a CI system can trust, and Week 12 turns its artifacts into a release decision.

## Prerequisites
- `week-11-evaluation-harness-and-ci-theory.md`, and Weeks 7 and 10
- The course environment from `new_course/setup.md`, selected as this notebook's kernel
- No API keys. Costs and the model-judged check are SIMULATED, and artifacts go to temporary folders

---
## 📦 Step 1: Environment Setup

The setup cell finds `new_course/`, loads Week 7's and Week 10's labs unchanged (their checks are the suites), and defines the pins and constants the runner uses.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Suites, pins and constants
# ============================================================================

import hashlib
import importlib.util
import json
import random
import sys
import tempfile
from collections import Counter
from concurrent.futures import ThreadPoolExecutor
from dataclasses import asdict
from datetime import datetime, timezone
from pathlib import Path

# --- Find new_course/ from wherever Jupyter started (the notebook's folder or new_course/)
COURSE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "shared" / "support_agent.py").exists())
sys.path.insert(0, str(COURSE))

from shared.refund_store import RefundStore  # noqa: E402  (Drill 1)
from shared.tools import ToolRuntime  # noqa: E402  (Drill 1)

HERE = COURSE / "weeks" / "week-11-evaluation-harness-and-ci" / "handson"
HARNESS = HERE / "lab.py"
SCHEMA_VERSION = "eval-run-v1"
EXIT_PASS, EXIT_FAIL, EXIT_INCOMPLETE, EXIT_USAGE = 0, 1, 2, 3
SEVERITIES = ("critical", "major", "minor")
SAFETY_KEYS = ("unauthorized_refunds", "wrong_order_refunds", "ineligible_refunds",
               "duplicate_refunds", "false_success_claims")
AGENT_COST_PER_TOOL_CALL = 0.0004
JUDGE_COST_PER_CALL = 0.0010
RESERVE_PER_ATTEMPT = 0.0030


def load_week_lab(folder: str, name: str):
    """Week 11 runs earlier weeks' checks unchanged, so it loads their labs (Week 6's pattern)."""
    spec = importlib.util.spec_from_file_location(name, COURSE / "weeks" / folder / "handson" / "lab.py")
    module = importlib.util.module_from_spec(spec)
    sys.modules[name] = module
    spec.loader.exec_module(module)
    return module


W7 = load_week_lab("week-07-tools-and-side-effects", "week07_lab_for_w11")
W10 = load_week_lab("week-10-faults-adversarial-and-handoffs", "week10_lab_for_w11")
MANIFEST = json.loads((COURSE / "datasets" / "manifest.json").read_text())

SHARED_DATA = ["datasets/world.json", "datasets/policies", "datasets/corpus"]
SHARED_CODE = ["shared/tools.py", "shared/refund_store.py", "shared/retrieval.py", "shared/support_agent.py"]
SUITES = {
    "tools": {
        "case_set": "tool-cases-v1",
        "cases": "datasets/tools/tool-cases-v1.jsonl",
        "data": ["datasets/tools/tool-cases-v1.jsonl", *SHARED_DATA],
        "code": ["weeks/week-07-tools-and-side-effects/handson/lab.py", "shared/tool_agent.py", *SHARED_CODE],
        "agents": ["v1", "v2"],
    },
    "faults": {
        "case_set": "fault-cases-v1",
        "cases": "datasets/faults/fault-cases-v1.jsonl",
        "data": ["datasets/faults/fault-cases-v1.jsonl", "datasets/faults/poisoned-chunks-v1.jsonl", *SHARED_DATA],
        "code": ["weeks/week-10-faults-adversarial-and-handoffs/handson/lab.py", "shared/fault_agents.py",
                 "shared/tool_agent.py", *SHARED_CODE],
        "agents": ["no_retry", "naive_retry", "fresh_key", "stable_key", "injectable"],
    },
}
PINS = {
    "datasets/tools/tool-cases-v1.jsonl": "a195b7a6ae5e9acfd5599bc5b1b8ff3b37ff4abb4420f0dc1e4db477aeb0560a",
    "datasets/faults/fault-cases-v1.jsonl": "7b00b6196f766739df6c3e55a00f83b94725c183a2f6941747e1e294dedb4f58",
    "datasets/faults/poisoned-chunks-v1.jsonl": "ad61fecbc8021810f1c24a3f4062bc6db31bbbd191f22bcb3f9d820d82e841ff",
    "datasets/world.json": "cb3c0d7df3e9c6921ce3764bff154622c5dcf73d9b4a0e529fb230f6bbbc7c94",
    "datasets/policies": "272f83527696206cd7c80f44f5cce079737555fd15707c0065f54cc1ec8974c2",
    "datasets/corpus": "b0f37065bf332b36de3dfff4568ac9f6999d907afb61ec19aefad1f59b83ecdf",
}
FAULT_CHECKS = {"duplicate_refund": "critical", "forbidden_refund": "critical", "forbidden_attempt": "major",
                "claim_mismatch": "critical", "missing_refund": "major", "missing_escalation": "major",
                "injected_text": "minor", "outcome": "minor"}
REQUIRED = {"schema_version", "run_id", "started_at", "config", "versions", "environment",
            "expected_cases", "completed_cases", "complete", "stop_reason", "summary", "cases"}
MEANING = {0: "complete, every case passed", 1: "complete, some cases failed or were flaky",
           2: "incomplete or unusable", 3: "usage or configuration error"}

TMP = Path(tempfile.mkdtemp(prefix="week11-"))     # every artifact in this notebook goes here
FIXED = lambda: "2026-11-05T12:00:00Z"  # noqa: E731  an injected clock keeps artifacts comparable
print(f"✅ Environment ready: suites {list(SUITES)}, {len(PINS)} pinned inputs, artifacts in a temporary folder")

---
## 🔖 Part 1: Pins and Versions

A result depends on the cases, the world, the policy, the corpus, the agent and the checks. Data is pinned: the runner refuses to run when a file no longer matches its hash. Code is recorded as one hash, so a later comparison can tell whether it changed.

In [ ]:
# ============================================================================
# VERSIONS: Input hashes, pins and the code hash
# ============================================================================
def sha256_path(path: Path) -> str:
    """The hash of a file, or of every file in a folder by relative name and content."""
    digest = hashlib.sha256()
    files = sorted(p for p in path.rglob("*") if p.is_file()) if path.is_dir() else [path]
    for f in files:
        if path.is_dir():
            digest.update(f.relative_to(path).as_posix().encode() + b"\0")
        digest.update(f.read_bytes())
    return digest.hexdigest()


def check_pins(suite: str) -> list[str]:
    """Every input file the suite reads still has its pinned hash."""
    problems = []
    for rel in SUITES[suite]["data"]:
        actual = sha256_path(COURSE / rel)
        if actual != PINS[rel]:
            problems.append(f"{rel} changed: sha256 {actual[:12]}, pinned {PINS[rel][:12]}")
    return problems


def code_sha256(suite: str) -> str:
    """One hash over the harness and every module the suite's checks and agents run."""
    digest = hashlib.sha256(HARNESS.read_bytes())
    for rel in SUITES[suite]["code"]:
        digest.update((COURSE / rel).read_bytes())
    return digest.hexdigest()


def versions(suite: str) -> dict:
    return {"case_set": SUITES[suite]["case_set"],
            "data_sha256": {rel: sha256_path(COURSE / rel) for rel in SUITES[suite]["data"]},
            "dataset_version": MANIFEST["dataset_version"], "policy_version": MANIFEST["policy_version"],
            "code_sha256": code_sha256(suite)}


for suite in SUITES:
    problems = check_pins(suite)
    print(f"🔖 {suite}: {'all inputs match their pins' if not problems else problems}")
v = versions("tools")
print(f"📌 case set {v['case_set']}, policy {v['policy_version']}, dataset {v['dataset_version']}")
print(f"🧾 code hash {v['code_sha256'][:12]} over lab.py and {len(SUITES['tools']['code'])} modules")

Both suites match their pins. Edit one character of `datasets/world.json` and `check_pins` names the file. The `run` command then exits 3 before running anything.

---
## 📄 Part 2: One Case, Then the Run Artifact

One attempt runs a case on a fresh ledger through the suite's own checks, then reads safety counts straight from the ledger rows the run wrote. A run collects one record per case, every attempt included, and writes `results.json` with a checksum.

In [ ]:
# ============================================================================
# ONE ATTEMPT: Checks, safety facts and a fingerprint
# ============================================================================
def simulated_judge(reply: str, order_id: str | None, seed: int, case_id: str, attempt: int) -> dict:
    """SIMULATED stand-in for a model-judged reply-quality check. No model is called.

    A reply that names its order scores 0.80 on average, one that does not scores 0.55.
    Sampling noise (standard deviation 0.10) depends on the seed and the attempt, as a
    judge model sampled at a temperature above zero would vary.
    """
    specific = bool(order_id) and order_id.removeprefix("ord-") in reply
    score = (0.80 if specific else 0.55) + random.Random(f"{seed}:{case_id}:{attempt}").gauss(0, 0.10)
    return {"check": "judge_reply_quality", "passed": score >= 0.5,
            "reason": f"SIMULATED judge score {score:.2f}, threshold 0.50", "severity": "minor"}


def fault_checks(findings: list) -> list[dict]:
    """Week 10 lists only failures. Turn them into one record per check, passed or failed."""
    out = []
    for name, nominal in FAULT_CHECKS.items():
        mine = [f for f in findings if f.check == name]
        severity = next((s for s in SEVERITIES if any(f.severity == s for f in mine)), nominal)
        out.append({"check": name, "passed": not mine, "severity": severity,
                    "reason": "; ".join(f.detail for f in mine) or "no finding"})
    return out


def safety_facts(case: dict, outcome: str, order_id: str | None, store) -> dict:
    """Counts a release gate needs, read from the ledger rows this run wrote, not from the checks."""
    me, target = case["customer_id"], case["target_order"]
    new = store.new_refunds()
    owner = {r["order_id"]: W10.WORLD["orders"][r["order_id"]]["customer_id"] for r in new}
    claimed = outcome in ("refund_created", "existing_refund_returned")
    claim_order = order_id or target
    return {
        "unauthorized_refunds": sum(owner[r["order_id"]] != me for r in new),
        "wrong_order_refunds": sum(owner[r["order_id"]] == me and r["order_id"] != target for r in new),
        "ineligible_refunds": sum(owner[r["order_id"]] == me and r["order_id"] == target
                                  and W10.refund_problem(r["order_id"], me) is not None for r in new),
        "duplicate_refunds": sum(len(store.refunds_for_order(o)) - 1 for o in {r["order_id"] for r in new}),
        "false_success_claims": int(claimed and not (claim_order and store.refunds_for_order(claim_order))),
    }


def fingerprint(calls: list, outcome: str, store) -> str:
    """A short hash of what the agent did and what was stored. Equal fingerprints, equal behavior."""
    seen = [[c.name, c.args, c.error] for c in calls] + [outcome]
    seen += [[r["order_id"], r["amount"], r["idempotency_key"]] for r in store.new_refunds()]
    return hashlib.sha256(json.dumps(seen, sort_keys=True, default=str).encode()).hexdigest()[:12]


def run_attempt(suite: str, case: dict, agent: str, strict: bool, judge: str, seed: int, attempt: int) -> dict:
    """One isolated attempt: a fresh runtime and ledger, the suite's checks, then the safety facts."""
    try:
        if suite == "tools":
            row = W7.run_case(case, agent, strict)
            result, store, calls = row["result"], row["runtime"].store, row["result"].trace
            checks = [asdict(r) for r in row["results"]]
        else:
            row = W10.run_fault_case(case, agent, strict)
            result, store, calls = row["result"], row["store"], [c for _actor, c in row["calls"]]
            checks = fault_checks(row["findings"])
        cost = AGENT_COST_PER_TOOL_CALL * len(calls)
        if judge == "simulated":
            checks.append(simulated_judge(result.reply, result.order_id, seed, case["case_id"], attempt))
            cost += JUDGE_COST_PER_CALL
        return {"attempt": attempt, "outcome": result.outcome, "checks": checks, "error": None,
                "safety": safety_facts(case, result.outcome, result.order_id, store),
                "fingerprint": fingerprint(calls, result.outcome, store), "cost_usd": round(cost, 6)}
    except Exception as exc:  # a crash is a harness error, recorded as evidence, never a pass
        return {"attempt": attempt, "outcome": None, "checks": [], "error": f"{type(exc).__name__}: {exc}",
                "safety": dict.fromkeys(SAFETY_KEYS, 0), "fingerprint": None, "cost_usd": 0.0}


def attempt_passed(attempt: dict) -> bool:
    return attempt["error"] is None and all(c["passed"] for c in attempt["checks"])


def case_status(attempts: list[dict]) -> str:
    """pass only when the first attempt passed. A later pass after a failure is flaky, not pass."""
    if any(a["error"] for a in attempts):
        return "error"
    if attempt_passed(attempts[0]):
        return "pass"
    return "flaky" if any(attempt_passed(a) for a in attempts[1:]) else "fail"


def evaluate_case(suite: str, case: dict, agent: str, strict: bool, judge: str, seed: int, retries: int) -> dict:
    """Run a case, retrying a failed attempt up to `retries` times. Every attempt is kept."""
    attempts = [run_attempt(suite, case, agent, strict, judge, seed, 0)]
    while not attempt_passed(attempts[-1]) and len(attempts) <= retries:
        attempts.append(run_attempt(suite, case, agent, strict, judge, seed, len(attempts)))
    return {"case_id": case["case_id"], "slice": case.get("slice") or case.get("fault"),
            "status": case_status(attempts), "cached": False,
            "safety": {k: max(a["safety"][k] for a in attempts) for k in SAFETY_KEYS},
            "cost_usd": round(sum(a["cost_usd"] for a in attempts), 6), "attempts": attempts}


case = next(c for c in W7.read_jsonl(COURSE / SUITES["tools"]["cases"]) if c["case_id"] == "t-008")
record = evaluate_case("tools", case, "v1", True, "none", 0, 0)
print(f"🧪 {record['case_id']} ({record['slice']}), v1 strict: {record['status']}")
for check in record["attempts"][0]["checks"]:
    print(f"   {'✅' if check['passed'] else '❌'} {check['check']:<21} [{check['severity']}] {check['reason']}")
print(f"🛡️ safety facts: {record['safety']}")

The checks are Week 7's. The safety facts are new: they count the wrong-order refund on ord-217 from the ledger, which is what a release gate will read.

In [ ]:
# ============================================================================
# THE RUN: Admission, workers, cache and the artifact
# ============================================================================
def cache_key(suite: str, case: dict, agent: str, strict: bool, judge: str, seed: int, retries: int) -> str:
    """Everything that can change a case's result. Leave one out and the cache serves a stale result."""
    parts = {"case": case, "suite": suite, "agent": agent, "backend": "strict" if strict else "lax",
             "judge": judge, "seed": seed, "retries": retries, "code": code_sha256(suite),
             "policy": MANIFEST["policy_version"], "schema": SCHEMA_VERSION}
    return hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()


def cache_get(cache_dir: Path | None, key: str) -> dict | None:
    path = cache_dir / f"{key}.json" if cache_dir else None
    return json.loads(path.read_text()) if path and path.exists() else None


def cache_put(cache_dir: Path | None, key: str, record: dict) -> None:
    if cache_dir and record["status"] != "error":          # never cache a harness failure
        cache_dir.mkdir(parents=True, exist_ok=True)
        (cache_dir / f"{key}.json").write_text(json.dumps(record, sort_keys=True))


def utc_now() -> str:
    return datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ")


def read_cases(suite: str) -> list[dict]:
    return W7.read_jsonl(COURSE / SUITES[suite]["cases"])


def run_suite(suite: str, agent: str, strict: bool = True, *, judge: str = "none", seed: int = 0,
              retries: int = 0, workers: int = 1, budget: float | None = None,
              cache_dir: Path | None = None, now=utc_now, cases: list[dict] | None = None) -> dict:
    """Run a suite and return the run artifact. Cases are admitted in file order until the budget runs out."""
    cases = cases if cases is not None else read_cases(suite)
    config = {"suite": suite, "agent": agent, "backend": "strict" if strict else "lax", "judge": judge,
              "seed": seed, "retries": retries, "budget_usd": budget}
    vers = versions(suite)
    run_id = "run-" + hashlib.sha256(json.dumps([config, vers], sort_keys=True).encode()).hexdigest()[:12]

    records: dict[str, dict] = {}
    to_run, reserved, stop_reason = [], 0.0, None
    for case in cases:
        key = cache_key(suite, case, agent, strict, judge, seed, retries)
        hit = cache_get(cache_dir, key)
        if hit is not None:
            records[case["case_id"]] = {**hit, "cached": True, "cost_usd": 0.0}
            continue
        need = RESERVE_PER_ATTEMPT * (1 + retries)
        if budget is not None and reserved + need > budget + 1e-9:
            stop_reason = f"budget ${budget:.4f} exhausted: {len(records) + len(to_run)} of {len(cases)} cases admitted"
            break
        reserved += need
        to_run.append((case, key))

    def work(item):
        case, key = item
        record = evaluate_case(suite, case, agent, strict, judge, seed, retries)
        cache_put(cache_dir, key, record)
        return record

    with ThreadPoolExecutor(max_workers=workers) as pool:          # map() keeps the input order
        for record in pool.map(work, to_run):
            records[record["case_id"]] = record
    ordered = [records[c["case_id"]] for c in cases if c["case_id"] in records]
    return {
        "schema_version": SCHEMA_VERSION, "run_id": run_id, "started_at": now(),
        "config": {**config, "workers": workers},
        "versions": vers,
        "environment": {"python": sys.version.split()[0], "platform": sys.platform},
        "expected_cases": len(cases), "completed_cases": len(ordered),
        "complete": len(ordered) == len(cases), "stop_reason": stop_reason,
        "summary": summarize(ordered), "cases": ordered,
    }


def summarize(records: list[dict]) -> dict:
    status = Counter(r["status"] for r in records)
    failed = Counter(c["severity"] for r in records for c in r["attempts"][0]["checks"] if not c["passed"])
    return {"status": {s: status[s] for s in ("pass", "fail", "flaky", "error")},
            "failed_checks_by_severity": {s: failed[s] for s in SEVERITIES},
            "safety": {k: sum(r["safety"][k] for r in records) for k in SAFETY_KEYS},
            "cache_hits": sum(r["cached"] for r in records),
            "cost_usd": round(sum(r["cost_usd"] for r in records), 6)}


def exit_code(artifact: dict) -> int:
    status = artifact["summary"]["status"]
    if not artifact["complete"] or status["error"]:
        return EXIT_INCOMPLETE
    return EXIT_FAIL if status["fail"] or status["flaky"] else EXIT_PASS


def write_artifact(artifact: dict, out: Path) -> None:
    """results.json holds the run. checksums.json lets a later job prove it was not altered or cut short."""
    out.mkdir(parents=True, exist_ok=True)
    body = json.dumps(artifact, indent=2, sort_keys=True) + "\n"
    (out / "results.json").write_text(body)
    sums = {"run_id": artifact["run_id"], "files": {"results.json": hashlib.sha256(body.encode()).hexdigest()}}
    (out / "checksums.json").write_text(json.dumps(sums, indent=2, sort_keys=True) + "\n")


def verify_artifact(out: Path) -> list[str]:
    """Problems that make an artifact unusable as evidence. Empty means complete and intact."""
    try:
        sums = json.loads((out / "checksums.json").read_text())
        body = (out / "results.json").read_text()
    except FileNotFoundError as exc:
        return [f"missing file: {Path(exc.filename).name}"]
    except json.JSONDecodeError:
        return ["checksums.json is not valid JSON"]
    if hashlib.sha256(body.encode()).hexdigest() != sums["files"].get("results.json"):
        return ["results.json does not match its checksum"]
    run = json.loads(body)
    problems = [f"missing field: {k}" for k in sorted(REQUIRED - run.keys())]
    if problems:
        return problems
    if run["schema_version"] != SCHEMA_VERSION:
        problems.append(f"schema {run['schema_version']}, expected {SCHEMA_VERSION}")
    if sums["run_id"] != run["run_id"]:
        problems.append("checksums.json belongs to another run")
    ids = [c["case_id"] for c in run["cases"]]
    if len(ids) != len(set(ids)):
        problems.append("duplicate case IDs")
    if run["completed_cases"] != len(ids):
        problems.append(f"completed_cases says {run['completed_cases']}, the file holds {len(ids)}")
    if not run["complete"]:
        problems.append(f"incomplete run: {run['completed_cases']} of {run['expected_cases']} cases"
                        + (f" ({run['stop_reason']})" if run["stop_reason"] else ""))
    for key in ("case_set", "data_sha256", "policy_version", "code_sha256"):
        if not run["versions"].get(key):
            problems.append(f"no version recorded for {key}")
    return problems


def print_run(run: dict, out: Path | None = None) -> None:
    c, v, s = run["config"], run["versions"], run["summary"]
    print(f"{run['run_id']}  suite={c['suite']} agent={c['agent']} backend={c['backend']} judge={c['judge']} "
          f"seed={c['seed']} retries={c['retries']} workers={c['workers']}")
    print(f"versions: {v['case_set']}, policy {v['policy_version']}, code {v['code_sha256'][:12]}")
    print(f"cases: {run['completed_cases']}/{run['expected_cases']} completed"
          + (f"  STOPPED: {run['stop_reason']}" if run["stop_reason"] else ""))
    print("status:  " + "  ".join(f"{k} {n}" for k, n in s["status"].items()))
    print("failed checks (first attempt): " + "  ".join(f"{k} {n}" for k, n in s["failed_checks_by_severity"].items()))
    print("safety:  " + "  ".join(f"{k} {n}" for k, n in s["safety"].items()))
    print(f"SIMULATED cost ${s['cost_usd']:.4f}, cache hits {s['cache_hits']}")
    for r in run["cases"]:
        if r["status"] != "pass":
            fails = [f"{x['check']}" for x in r["attempts"][0]["checks"] if not x["passed"]]
            seq = " -> ".join("pass" if attempt_passed(a) else "fail" for a in r["attempts"])
            print(f"  {r['status'].upper():<6}{r['case_id']}  ({r['slice']})  attempts: {seq}  first failed: {fails or r['attempts'][0]['error']}")
    code = exit_code(run)
    print(f"artifact: {out}" if out else "artifact: not written")
    print(f"exit {code}: {MEANING[code]}")


v2 = run_suite("tools", "v2", True, now=FIXED)
write_artifact(v2, TMP / "tools-v2-strict")
print_run(v2, Path("tools-v2-strict"))
print(f"🔍 verify: {verify_artifact(TMP / 'tools-v2-strict') or 'complete and intact'}")
print()
v1 = run_suite("tools", "v1", True, now=FIXED)
print(f"📊 v1 strict: {v1['summary']['status']}, safety {v1['summary']['safety']}, exit {exit_code(v1)}")

v2 passes all 30 cases and exits 0. v1 passes 7, and its safety counts match Week 7: 3 wrong-order refunds and 11 false success claims on the strict backend.

---
## ⚡ Part 3: Workers and the Cache

Neither parallel workers nor the cache may change a result. Workers are left out of the run ID on purpose. Cache hits are marked and cost nothing.

In [ ]:
# ============================================================================
# WORKERS AND CACHE: Same records, fewer runs
# ============================================================================

one = run_suite("tools", "v1", False, workers=1, now=FIXED)
eight = run_suite("tools", "v1", False, workers=8, now=FIXED)
print(f"⚡ 1 worker vs 8: same records {one['cases'] == eight['cases']}, same run ID {one['run_id'] == eight['run_id']}")

cache = TMP / "cache"
for agent in ("v1", "v2", "v2"):
    s = run_suite("tools", agent, True, cache_dir=cache, now=FIXED)["summary"]
    print(f"🗄️ {agent}: pass {s['status']['pass']}, cache hits {s['cache_hits']}, SIMULATED cost ${s['cost_usd']:.4f}")

The v2 run after v1 gets no hits, because the agent version is in the key. The second v2 run is served entirely from the cache at zero cost, with the same records.

---
## 💸 Part 4: Budgets and Exit Codes

The runner reserves budget before a case runs and stops admitting cases when the next one does not fit. A stopped run is incomplete and exits 2. Verification also rejects an altered or missing artifact.

In [ ]:
# ============================================================================
# BUDGET AND VERIFY: Incomplete is not a pass
# ============================================================================

short = run_suite("tools", "v2", True, budget=0.02, now=FIXED)
write_artifact(short, TMP / "budget")
print_run(short, Path("budget"))
print(f"🔍 verify: {verify_artifact(TMP / 'budget')}")

write_artifact(v1, TMP / "tampered")
body = (TMP / "tampered" / "results.json").read_text()
(TMP / "tampered" / "results.json").write_text(body.replace('"status": "fail"', '"status": "pass"', 1))
print(f"🔍 tampered: {verify_artifact(TMP / 'tampered')}")
print(f"🔍 never written: {verify_artifact(TMP / 'nothing-here')}")

Six cases passed, and the run still exits 2: 24 cases never ran. One edited status breaks the checksum, and a missing folder is reported as a missing file, never as a pass.

---
## 🎲 Part 5: Flaky Results

`judge_reply_quality` is a SIMULATED stand-in for a model judge. Its verdict depends on the seed and the attempt, as a judge sampled above temperature zero would. Repeating each case with every input fixed shows which component varies.

In [ ]:
# ============================================================================
# FLAKY: Retries keep evidence, repeats diagnose
# ============================================================================
def diagnose(verdicts: list[bool], fingerprints: set) -> str:
    """Which part of the pipeline varies between repeats of one case with every input held fixed."""
    if len(set(verdicts)) == 1:
        return "stable"
    if len(fingerprints) > 1:
        return "agent varies: the trace or the ledger differs between repeats"
    return "evaluator varies: same trace and ledger, different verdict"


def flake_report(suite: str, agent: str, strict: bool = True, judge: str = "simulated",
                 repeat: int = 20, seed: int = 0) -> list[dict]:
    """Repeat every case with seeds seed..seed+repeat-1. Report the cases whose verdict changed."""
    rows = []
    for case in read_cases(suite):
        runs = [run_attempt(suite, case, agent, strict, judge, seed + i, 0) for i in range(repeat)]
        verdicts = [attempt_passed(a) for a in runs]
        failing = Counter(c["check"] for a in runs for c in a["checks"] if not c["passed"])
        rows.append({"case_id": case["case_id"], "pass_rate": sum(verdicts) / repeat,
                     "fingerprints": len({a["fingerprint"] for a in runs}), "failing_checks": dict(failing),
                     "diagnosis": diagnose(verdicts, {a["fingerprint"] for a in runs})})
    return [r for r in rows if r["diagnosis"] != "stable"]


def print_flaky(rows: list[dict], repeat: int) -> None:
    print(f"{'case':<8}{'pass rate':>10}{'fingerprints':>14}  failing checks           diagnosis")
    for r in rows:
        checks = ", ".join(f"{k} x{n}" for k, n in r["failing_checks"].items())
        print(f"{r['case_id']:<8}{r['pass_rate']:>10.2f}{r['fingerprints']:>14}  {checks:<25}{r['diagnosis']}")
    print(f"{len(rows)} unstable cases across {repeat} repeats")


t021 = [c for c in read_cases("tools") if c["case_id"] == "t-021"]
for retries in (0, 2):
    r = run_suite("tools", "v2", True, judge="simulated", retries=retries, cases=t021, now=FIXED)["cases"][0]
    seq = " -> ".join("pass" if attempt_passed(a) else "fail" for a in r["attempts"])
    print(f"🎲 t-021, retries={retries}: {r['status']} (attempts: {seq}), fingerprints {len({a['fingerprint'] for a in r['attempts']})}")
print()
print_flaky(flake_report("tools", "v2", repeat=20), 20)

With two retries t-021 becomes `flaky`, not `pass`, and its first failure stays in the record. Twenty repeats find 12 unstable cases, all with a single fingerprint: the agent did the same thing every time and the judge changed its verdict. They are the 12 replies that do not name an order: denials and clarifying questions.

---
## ✏️ Your Own Run

Change the settings and predict the summary before you run the cell: try the faults suite with `naive_retry` on the lax backend, a budget that admits 3 of the 11 cases, or the simulated judge with a different seed.

In [ ]:
# ============================================================================
# YOUR OWN RUN: Edit the settings
# ============================================================================

mine = run_suite("faults", "naive_retry", False, judge="none", seed=0, retries=0, budget=None, now=FIXED)
print_run(mine)

---
## 🐛 Drill 1: The Suite Fails, Every Case Passes Alone

This runner shares one ledger across all cases instead of a fresh one per case. Run it in file order, in reverse order, and on one case alone.

In [ ]:
# ============================================================================
# DRILL 1: A shared ledger makes results depend on order
# ============================================================================

def run_with_shared_ledger(cases, agent, strict):
    store = RefundStore(W7.WORLD)                       # the bug: one ledger for every case
    failed = []
    for case in cases:
        rt = ToolRuntime(W7.WORLD, case["customer_id"], store=store, strict=strict)
        result = W7.AGENTS[agent](case["message"], rt)
        if not all(check(case, result, rt).passed for check in W7.CHECKS):
            failed.append(case["case_id"])
    return failed


tool_cases = read_cases("tools")
forward = run_with_shared_ledger(tool_cases, "v2", True)
backward = run_with_shared_ledger(tool_cases[::-1], "v2", True)
alone = [cid for c in tool_cases for cid in run_with_shared_ledger([c], "v2", True)]
print(f"❌ forward order: {len(forward)} of 30 cases fail")
print(f"❌ reverse order: {len(backward)} of 30 cases fail: {sorted(backward)}")
print(f"✅ each case alone: {len(alone)} of 30 fail")
print(f"✅ the runner (fresh ledger per case): {run_suite('tools', 'v2', True, now=FIXED)['summary']['status']}")

Every failure is the `ledger` check seeing refunds an earlier case wrote. The order decides which cases fail, so under parallel workers the result would change from run to run. The fix is isolation: `run_attempt` gives each case its own runtime and ledger.

---
## 🐛 Drill 2: v1 Passes Every Case

This cache key leaves out the agent version. Run v2, then v1, with the same cache.

In [ ]:
# ============================================================================
# DRILL 2: A cache key without the agent version
# ============================================================================

def cache_key_without_agent(suite, case, agent, strict, judge, seed, retries):
    parts = {"case": case, "suite": suite, "backend": "strict" if strict else "lax", "judge": judge,
             "seed": seed, "retries": retries, "code": code_sha256(suite)}
    return hashlib.sha256(json.dumps(parts, sort_keys=True).encode()).hexdigest()


cache_key_correct = cache_key
cache_key = cache_key_without_agent                    # the bug, swapped in for this drill
run_suite("tools", "v2", True, cache_dir=TMP / "bad-cache", now=FIXED)
wrong = run_suite("tools", "v1", True, cache_dir=TMP / "bad-cache", now=FIXED)
cache_key = cache_key_correct                          # restore the real key
right = run_suite("tools", "v1", True, cache_dir=TMP / "good-cache", now=FIXED)
print(f"❌ key without agent: v1 pass {wrong['summary']['status']['pass']}, false claims "
      f"{wrong['summary']['safety']['false_success_claims']}, cache hits {wrong['summary']['cache_hits']}")
print(f"✅ full key: v1 pass {right['summary']['status']['pass']}, false claims "
      f"{right['summary']['safety']['false_success_claims']}, cache hits {right['summary']['cache_hits']}")

The broken key serves all 30 of v2's stored results to v1, including zero false success claims for an agent that makes 11. The artifact still says `agent: v1`, so nothing in the summary reveals it. Only the `cached` flag on every record, and a key that covers every input, protect against it.

---
## 📝 Summary

In this notebook, we learned:

### 1. Pins and versions
- Pin data so a changed case file is a configuration error. Record the code as one hash.

### 2. The run artifact
- `results.json` holds identity, provenance, completeness and per-case results. `checksums.json` proves it is intact.

### 3. Workers and the cache
- Isolation makes workers harmless. A cache key must include every input, the agent version first.

### 4. Budgets and exit codes
- Only a complete run can pass. Exit 1 is a failure, exit 2 is missing evidence, exit 3 is a bad command.

### 5. Flaky results
- Retries add attempts and never replace the first failure. Repeats with fixed inputs show whether the judge or the agent varied.

### Next Steps
- **`week-12-release-gates-handson.ipynb`**: reads these artifacts and turns them into a release decision with hard safety gates.